# 02. Univariate EDA & Distribution Diagnostics

How to inspect distributions across feature types and decide whether to transform, scale, or handle zero-inflation.


## 1. Objective
Demonstrate how to analyze single-feature distributions and make justified decisions regarding transformations, zero-inflation, and scaling.


## 2. Dataset & Decision Context
- **Dataset**: Used Car Market (`used_cars.csv`)
- **Target**: `selling_price` (Regression)
- **Features to Investigate**: `selling_price`, `mileage`, `engine_cc`, `brand`, `owner_count`


## 3. What Should I Check?

| Check | Why |
|---|---|
| **Target Distribution (`selling_price`)** | Linear regression assumes normally distributed residuals; a skewed target creates heteroscedastic error |
| **Feature Skewness (`mileage`)** | Extreme values dominate distance metrics (KNN/KMeans) and pull linear regression slopes |
| **Discrete Counts (`owner_count`)** | Determines if an integer column is continuous or ordinal |
| **Zero-Inflation / Boundary Truncation** | Features with clumps at zero require specialized dual-feature engineering |


## 4. Technique Breakdown

```
WHAT: Univariate Statistical & Visual Analysis (Histogram, KDE, Boxplot, Skewness, Quantiles)
WHY: Identifies non-normality, extreme values, multi-modality, and zero clumps
WHEN: Mandatory for every continuous and discrete feature
WHEN NOT: Never rely purely on mean and standard deviation for skewed features
HOW: Inspect histogram + boxplot, compute Fisher-Pearson skewness, test log1p
WHAT TO LOOK FOR: Skew > 1.0, long right tails, bimodal peaks
WHAT ACTION: Apply np.log1p() to right-skewed positive features; test power transforms
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

df = pd.read_csv('../datasets/used_cars/used_cars.csv')
print(f"Used Cars Dataset: {df.shape[0]:,} rows | {df.shape[1]} columns")
df[['selling_price', 'mileage', 'engine_cc', 'owner_count']].describe()


## 5. Diagnosing Continuous Skewness: Target & Mileage


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Raw Selling Price
sns.histplot(df['selling_price'], kde=True, ax=axes[0, 0], color='#2b5c8f', bins=40)
axes[0, 0].set_title(f"Raw Selling Price (Skew: {df['selling_price'].skew():.2f})")
axes[0, 0].set_xlabel("Price ($)")

# 2. Log-Transformed Selling Price
log_price = np.log1p(df['selling_price'])
sns.histplot(log_price, kde=True, ax=axes[0, 1], color='#27ae60', bins=40)
axes[0, 1].set_title(f"Log1p(Selling Price) (Skew: {log_price.skew():.2f})")
axes[0, 1].set_xlabel("log(Price + 1)")

# 3. Raw Mileage
sns.histplot(df['mileage'], kde=True, ax=axes[1, 0], color='#d95f02', bins=40)
axes[1, 0].set_title(f"Raw Mileage (Skew: {df['mileage'].skew():.2f})")
axes[1, 0].set_xlabel("Mileage (Miles)")

# 4. Log-Transformed Mileage (excluding non-positive anomalies)
valid_mileage = df['mileage'][df['mileage'] > 0]
log_mileage = np.log1p(valid_mileage)
sns.histplot(log_mileage, kde=True, ax=axes[1, 1], color='#8e44ad', bins=40)
axes[1, 1].set_title(f"Log1p(Mileage) (Skew: {log_mileage.skew():.2f})")
axes[1, 1].set_xlabel("log(Mileage + 1)")

plt.tight_layout()
plt.show()


## 6. Before / After Transformation Quantification


In [ ]:
comparison = pd.DataFrame({
    'Feature': ['selling_price', 'mileage'],
    'Raw_Skew': [df['selling_price'].skew(), df['mileage'].skew()],
    'Log1p_Skew': [np.log1p(df['selling_price']).skew(), np.log1p(df['mileage'][df['mileage'] > 0]).skew()],
    'Raw_Kurtosis': [df['selling_price'].kurtosis(), df['mileage'].kurtosis()],
    'Log1p_Kurtosis': [np.log1p(df['selling_price']).kurtosis(), np.log1p(df['mileage'][df['mileage'] > 0]).kurtosis()]
})
comparison


## 7. Categorical & Discrete Frequency Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Discrete: Owner Count
sns.countplot(data=df, x='owner_count', ax=axes[0], color='#2b5c8f')
axes[0].set_title('Owner Count Distribution (Discrete)')

# Categorical: Brand
brand_order = df['brand'].value_counts().index
sns.countplot(data=df, y='brand', order=brand_order, ax=axes[1], color='#27ae60')
axes[1].set_title('Brand Distribution (Categorical)')

plt.tight_layout()
plt.show()


## 8. Interpretation & Decision Log

### What did we find?
1. `selling_price` is strongly right-skewed (skew = 2.45). Transforming with $\log(x+1)$ reduces skew to 0.12, converting a heavy Pareto tail into a near-Gaussian distribution.
2. `mileage` contains a long right tail with vehicle commuters up to 350,000 miles. Log transformation compresses the variance from kurtosis = 8.4 down to 0.28.
3. `owner_count` has discrete values 1 to 4 with natural monotonic meaning $\rightarrow$ treat as numerical or ordinal.

### Explicit Decision
> [!IMPORTANT]
> **Decision Rule**:
> - **Because** `selling_price` is the target of a regression task and is heavily right-skewed, we **will** train linear regression and neural network models on $\log(\text{selling\_price})$ and invert predictions using $\exp(y) - 1$.
> - **Because** tree-based models (XGBoost/RandomForest) are invariant to monotonic transformations, we **do not need** log transforms for trees, but will test them for linear models.


## 9. Decision Table: Univariate Distributions

| Distribution Shape | Diagnostic Trigger | Recommended Transformation | Models Affected |
|---|---|---|---|
| **Right-Skewed ($x \ge 0$)** | Skew $> 1.0$, long right tail | `np.log1p(x)` or `Yeo-Johnson` | Linear Regression, Logistic, KNN, SVM, NN |
| **Heavy Outliers (Legitimate)** | Kurtosis $> 3.0$, extreme IQR spread | `RobustScaler` (Median/IQR) | Distance-based models (KNN, KMeans, SVM) |
| **Zero-Inflated** | $> 30\%$ values are exactly 0 | Dual feature: `is_zero` flag + `log1p(x)` | All model types |
| **Uniform / Bounded** | Fixed min/max range | `MinMaxScaler` into $[0, 1]$ | Neural Networks, Image/Signal models |
